In [129]:
import pandas as pd
import numpy as np
from datetime import datetime
import csv

# Loading Data

In [113]:
# Load datasets
print(f"Starting process at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Loading existing features and batting data...")

# Load features dataset
features_df = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\processed_data\all_league_bat_features.csv')
features_df['date'] = pd.to_datetime(features_df['date'])

# Load original batting data
batting_df = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\mbm\all_league_bat_final.csv')
batting_df['date'] = pd.to_datetime(batting_df['date'])


Starting process at 2025-04-13 04:43:30
Loading existing features and batting data...


In [114]:
features_df[features_df.career_avg.isnull()].shape

(8390, 35)

# Sorting the data

In [115]:
# Sort both dataframes by date
features_df.sort_values(['batter_id', 'date', 'inns'], inplace=True)
batting_df.sort_values(['batter_id', 'date', 'inns'], inplace=True)

# Convert isOut to numeric format in batting data
batting_df['isOut'] = batting_df['isOut'].replace('DNB', 'False')
batting_df['isOut'] = batting_df['isOut'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

# Add boundaries column to batting data
batting_df['boundaries'] = batting_df['fours'] + batting_df['sixes']

# Filling NULL values for league debuts

In [116]:
def fill_league_debut_nulls(features_df, batting_df):
    """Fill NULL values in features dataframe for league debuts using prior performance data.

    Args:
        features_df: DataFrame containing batting features with NULL values
        batting_df: DataFrame containing original batting performance data

    Returns:
        Updated features DataFrame with filled NULL values
    """
    # Create a copy to avoid modifying the original
    features_df = features_df.copy()

    # Get unique player IDs
    unique_players = features_df['batter_id'].unique()
    print(f"Processing {len(unique_players)} unique players")

    # Counter for tracking updates
    updates_made = 0
    league_debuts_found = 0

    # Process each player
    for i, player_id in enumerate(unique_players):
        if (i+1) % 100 == 0 or i+1 == len(unique_players):
            print(f"Processed {i+1}/{len(unique_players)} players")

        # Get player data from features
        player_features = features_df[features_df['batter_id'] == player_id].copy()

        # Skip if player only has one entry (absolute debut)
        if len(player_features) <= 1:
            continue

        # Get all player's batting records
        player_batting = batting_df[batting_df['batter_id'] == player_id].copy()

        # Check for league debuts (nulls in career stats)
        league_debuts = player_features[pd.isna(player_features['career_avg'])]

        # Skip if no league debuts found after absolute debut
        if len(league_debuts) <= 1:  # First one is absolute debut, which should stay NULL
            continue

        league_debuts_found += len(league_debuts) - 1  # Don't count absolute debut

        # Process each league debut (skip absolute debut)
        for idx, debut in league_debuts.iterrows():
            # Skip the absolute debut (first match ever)
            if debut['date'] == player_features['date'].min():
                continue

            # Get all matches before this date
            prior_matches = player_batting[player_batting['date'] < debut['date']]

            if len(prior_matches) > 0:
                # Calculate career stats based on prior matches
                total_runs = prior_matches['runs'].sum()
                total_dismissals = prior_matches['isOut'].sum()
                total_matches = len(prior_matches)
                total_balls = prior_matches['balls'].sum()
                total_boundaries = prior_matches['boundaries'].sum()

                # Calculate career stats
                career_avg = total_runs / max(total_dismissals, 1)
                career_rpm = total_runs / total_matches
                career_sr = (total_runs * 100) / max(total_balls, 1)
                career_bpm = total_boundaries / total_matches
                career_bfpm = total_balls / total_matches

                # Update the features dataframe
                features_df.loc[idx, 'career_avg'] = career_avg
                features_df.loc[idx, 'career_rpm'] = career_rpm
                features_df.loc[idx, 'career_sr'] = career_sr
                features_df.loc[idx, 'career_bpm'] = career_bpm
                features_df.loc[idx, 'career_bfpm'] = career_bfpm

                updates_made += 1

                # If innings is the same, we can also update innings stats
                prior_inns_matches = prior_matches[prior_matches['inns'] == debut['inns']]
                if len(prior_inns_matches) > 0:
                    inns_runs = prior_inns_matches['runs'].sum()
                    inns_dismissals = prior_inns_matches['isOut'].sum()
                    inns_matches = len(prior_inns_matches)
                    inns_balls = prior_inns_matches['balls'].sum()
                    inns_boundaries = prior_inns_matches['boundaries'].sum()

                    inns_avg = inns_runs / max(inns_dismissals, 1)
                    inns_rpm = inns_runs / inns_matches
                    inns_sr = (inns_runs * 100) / max(inns_balls, 1)
                    inns_bpm = inns_boundaries / inns_matches
                    inns_bfpm = inns_balls / inns_matches

                    features_df.loc[idx, 'inns_avg'] = inns_avg
                    features_df.loc[idx, 'inns_rpm'] = inns_rpm
                    features_df.loc[idx, 'inns_sr'] = inns_sr
                    features_df.loc[idx, 'inns_bpm'] = inns_bpm
                    features_df.loc[idx, 'inns_bfpm'] = inns_bfpm

                # VENUE STATS - UPDATED LOGIC
                # For new venue, use overall stats from all prior matches
                prior_venue_matches = prior_matches[prior_matches['venue'] == debut['venue']]
                if len(prior_venue_matches) > 0:
                    # Player has played at this venue before - use venue-specific stats
                    venue_runs = prior_venue_matches['runs'].sum()
                    venue_dismissals = prior_venue_matches['isOut'].sum()
                    venue_matches = len(prior_venue_matches)
                    venue_balls = prior_venue_matches['balls'].sum()
                    venue_boundaries = prior_venue_matches['boundaries'].sum()
                else:
                    # First time at this venue - use overall stats
                    venue_runs = total_runs
                    venue_dismissals = total_dismissals
                    venue_matches = total_matches
                    venue_balls = total_balls
                    venue_boundaries = total_boundaries

                ven_avg = venue_runs / max(venue_dismissals, 1)
                ven_rpm = venue_runs / venue_matches
                ven_sr = (venue_runs * 100) / max(venue_balls, 1)
                ven_bpm = venue_boundaries / venue_matches
                ven_bfpm = venue_balls / venue_matches

                features_df.loc[idx, 'ven_avg'] = ven_avg
                features_df.loc[idx, 'ven_rpm'] = ven_rpm
                features_df.loc[idx, 'ven_sr'] = ven_sr
                features_df.loc[idx, 'ven_bpm'] = ven_bpm
                features_df.loc[idx, 'ven_bfpm'] = ven_bfpm

                # SEASON STATS - UPDATED LOGIC
                if 'season' in features_df.columns and not pd.isna(debut['season']):
                    season = debut['season']
                    debut_league = debut['league']

                    # Check if player has played in this league before
                    prior_league_matches = prior_matches[prior_matches['league'] == debut_league]

                    if len(prior_league_matches) > 0:
                        # If this is not the first season in this league
                        # Use the most recent previous season in this league
                        if isinstance(season, (int, float)):
                            # Get all seasons played in this league
                            league_seasons = sorted(prior_league_matches['date'].dt.year.unique())
                            # Find the most recent previous season
                            prev_seasons = [s for s in league_seasons if s < season]
                            if prev_seasons:
                                prev_season = max(prev_seasons)
                                # Get matches from previous season in this league
                                prev_season_matches = prior_league_matches[prior_league_matches['date'].dt.year == prev_season]

                                if len(prev_season_matches) > 0:
                                    season_runs = prev_season_matches['runs'].sum()
                                    season_dismissals = prev_season_matches['isOut'].sum()
                                    season_matches = len(prev_season_matches)
                                    season_balls = prev_season_matches['balls'].sum()
                                    season_boundaries = prev_season_matches['boundaries'].sum()
                                else:
                                    # Fallback to all matches in this league
                                    season_runs = prior_league_matches['runs'].sum()
                                    season_dismissals = prior_league_matches['isOut'].sum()
                                    season_matches = len(prior_league_matches)
                                    season_balls = prior_league_matches['balls'].sum()
                                    season_boundaries = prior_league_matches['boundaries'].sum()
                            else:
                                # No previous season, use all matches in this league
                                season_runs = prior_league_matches['runs'].sum()
                                season_dismissals = prior_league_matches['isOut'].sum()
                                season_matches = len(prior_league_matches)
                                season_balls = prior_league_matches['balls'].sum()
                                season_boundaries = prior_league_matches['boundaries'].sum()
                    else:
                        # First season in this league, use data from all leagues for this season
                        if isinstance(season, (int, float)):
                            same_season_matches = prior_matches[prior_matches['date'].dt.year == season]

                            if len(same_season_matches) > 0:
                                # Use matches from the same season across all leagues
                                season_runs = same_season_matches['runs'].sum()
                                season_dismissals = same_season_matches['isOut'].sum()
                                season_matches = len(same_season_matches)
                                season_balls = same_season_matches['balls'].sum()
                                season_boundaries = same_season_matches['boundaries'].sum()
                            else:
                                # No matches in this season, use all prior matches
                                season_runs = total_runs
                                season_dismissals = total_dismissals
                                season_matches = total_matches
                                season_balls = total_balls
                                season_boundaries = total_boundaries
                        else:
                            # If season is not a number, use all prior matches
                            season_runs = total_runs
                            season_dismissals = total_dismissals
                            season_matches = total_matches
                            season_balls = total_balls
                            season_boundaries = total_boundaries

                    # Calculate and update season stats
                    ssn_avg = season_runs / max(season_dismissals, 1)
                    ssn_rpm = season_runs / season_matches
                    ssn_sr = (season_runs * 100) / max(season_balls, 1)
                    ssn_bpm = season_boundaries / season_matches
                    ssn_bfpm = season_balls / season_matches

                    features_df.loc[idx, 'ssn_avg'] = ssn_avg
                    features_df.loc[idx, 'ssn_rpm'] = ssn_rpm
                    features_df.loc[idx, 'ssn_sr'] = ssn_sr
                    features_df.loc[idx, 'ssn_bpm'] = ssn_bpm
                    features_df.loc[idx, 'ssn_bfpm'] = ssn_bfpm

                # Update last5 stats if we have enough prior matches
                if 'last5_avg' in features_df.columns:
                    # Use all available prior matches, up to 5
                    last5_matches = prior_matches.sort_values('date', ascending=False).head(min(5, len(prior_matches)))

                    last5_runs = last5_matches['runs'].sum()
                    last5_dismissals = last5_matches['isOut'].sum()
                    last5_matches_count = len(last5_matches)
                    last5_balls = last5_matches['balls'].sum()
                    last5_boundaries = last5_matches['boundaries'].sum()

                    last5_avg = last5_runs / max(last5_dismissals, 1)
                    last5_rpm = last5_runs / last5_matches_count
                    last5_sr = (last5_runs * 100) / max(last5_balls, 1)
                    last5_bpm = last5_boundaries / last5_matches_count
                    last5_bfpm = last5_balls / last5_matches_count

                    features_df.loc[idx, 'last5_avg'] = last5_avg
                    features_df.loc[idx, 'last5_sr'] = last5_sr
                    features_df.loc[idx, 'last5_rpm'] = last5_rpm
                    features_df.loc[idx, 'last5_bfpm'] = last5_bfpm
                    features_df.loc[idx, 'last5_bpm'] = last5_bpm

    print(f"\nFound {league_debuts_found} league debuts (after absolute debut)")
    print(f"Made {updates_made} updates to fill NULL values")

    return features_df

In [117]:
updated_features_df = fill_league_debut_nulls(features_df, batting_df)


Processing 4708 unique players
Processed 100/4708 players
Processed 200/4708 players
Processed 300/4708 players
Processed 400/4708 players
Processed 500/4708 players
Processed 600/4708 players
Processed 700/4708 players
Processed 800/4708 players
Processed 900/4708 players
Processed 1000/4708 players
Processed 1100/4708 players
Processed 1200/4708 players
Processed 1300/4708 players
Processed 1400/4708 players
Processed 1500/4708 players
Processed 1600/4708 players
Processed 1700/4708 players
Processed 1800/4708 players
Processed 1900/4708 players
Processed 2000/4708 players
Processed 2100/4708 players
Processed 2200/4708 players
Processed 2300/4708 players
Processed 2400/4708 players
Processed 2500/4708 players
Processed 2600/4708 players
Processed 2700/4708 players
Processed 2800/4708 players
Processed 2900/4708 players
Processed 3000/4708 players
Processed 3100/4708 players
Processed 3200/4708 players
Processed 3300/4708 players
Processed 3400/4708 players
Processed 3500/4708 player

# Checking for NULL filled values

In [118]:
updated_features_df.sample()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
13916,1185850,2019-09-17,1,Ali Khan,927119,Trinbago Knight Riders,St Kitts and Nevis Patriots,Warner Park,CPL,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [119]:
# Find a player with NULL values across leagues for demonstration
example_player_id = 959767  # Example fallback
print(f"Checking for player ID {example_player_id} or finding a suitable example...")

player_data = features_df[features_df['batter_id'] == example_player_id].sort_values('date')

if player_data.empty or player_data['career_avg'].isna().sum() <= 1:
    # Find a player who has played in multiple leagues and has NULL values
    league_counts = features_df.groupby('batter_id')['league'].nunique()
    multi_league_players = league_counts[league_counts > 1].index.tolist()

    for pid in multi_league_players:
        p_data = features_df[features_df['batter_id'] == pid].sort_values('date')
        null_counts = p_data['career_avg'].isna().sum()

        if null_counts > 1:  # More than just the first match
            example_player_id = pid
            player_data = p_data
            player_name = player_data['batter'].iloc[0] if 'batter' in player_data.columns else f"ID: {pid}"
            print(f"Found example player: {player_name} (ID: {pid})")
            break

# Get the leagues the player has played in
player_leagues = player_data['league'].unique()

# Find a league debut (not the first appearance overall)
first_date = player_data['date'].min()
league_debuts = player_data[pd.isna(player_data['career_avg']) & (player_data['date'] > first_date)]

if not league_debuts.empty:
    debut_idx = league_debuts.index[0]
    debut_league = league_debuts.iloc[0]['league']
    debut_date = league_debuts.iloc[0]['date']

    # Display before filling
    print("\n--- BEFORE FILLING ---")
    print(f"Row with NULL value (first {debut_league} match):")
    display(features_df.loc[[debut_idx], ['date', 'league', 'ven_avg', 'ven_sr', 'ven_bpm', 'ven_bfpm']])

    # Show prior data from other leagues
    prior_leagues = player_data[(player_data['date'] < debut_date) & (player_data['league'] != debut_league)]
    if not prior_leagues.empty:
        print(f"\nPrior data from {prior_leagues.iloc[0]['league']} before {debut_league} debut:")
        display(prior_leagues[['date', 'league', 'ven_avg', 'ven_sr', 'ven_bpm', 'ven_bfpm']])

    # Display after filling
    print("\n--- AFTER FILLING ---")
    print(f"Row with filled value (first {debut_league} match):")
    display(updated_features_df.loc[[debut_idx], ['date', 'league', 'ven_avg', 'ven_sr', 'ven_bpm', 'ven_bfpm']])

    print("\n✅ Success! The NULL values have been filled with the player's career stats from prior leagues.")
else:
    print("❌ Could not find a suitable example of a league debut for demonstration.")


Checking for player ID 959767 or finding a suitable example...

--- BEFORE FILLING ---
Row with NULL value (first T20IS match):


,date,league,ven_avg,ven_sr,ven_bpm,ven_bfpm
116202,2021-09-01,T20IS,NaN,NaN,NaN,NaN



Prior data from SSM before T20IS debut:


,date,league,ven_avg,ven_sr,ven_bpm,ven_bfpm
116180,2019-01-27,SSM,NaN,NaN,NaN,NaN
116181,2019-02-01,SSM,NaN,NaN,NaN,NaN
116182,2019-02-09,SSM,NaN,NaN,NaN,NaN
116183,2019-12-15,SSM,6.000000,66.666667,0.000000,9.000000
116184,2019-12-18,SSM,0.000000,0.000000,0.000000,1.000000
116185,2019-12-21,SSM,4.500000,112.500000,0.500000,4.000000
116186,2019-12-27,SSM,NaN,NaN,NaN,NaN
116187,2019-12-30,SSM,NaN,NaN,NaN,NaN
116188,2020-01-01,SSM,NaN,NaN,NaN,NaN
116189,2020-01-03,SSM,4.500000,112.500000,0.333333,2.666667



--- AFTER FILLING ---
Row with filled value (first T20IS match):


,date,league,ven_avg,ven_sr,ven_bpm,ven_bfpm
116202,2021-09-01,T20IS,17.117647,135.348837,1.727273,9.772727



✅ Success! The NULL values have been filled with the player's career stats from prior leagues.


In [120]:
updated_features_df[(updated_features_df.batter_id == 959767 ) & (updated_features_df.venue == 'McLean Park')]

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
116195,1233998,2020-12-27,1,R Ravindra,959767,Wellington,Central Districts,McLean Park,SSM,16.25,...,19.333333,16.571429,141.463415,2.0,11.714286,14.5,123.404255,11.6,9.4,1.4


In [121]:
batting_df[batting_df.batter_id == 959767]

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,fours,sixes,sr,isOut,wicketType,bowler,bowler_id,venue,league,boundaries
69777,1160520,2019-01-27,1,R Ravindra,959767,Wellington,Otago,0,0,0,0,0.00,0,DNB,NaN,NaN,University Oval,SSM,0
69806,1160521,2019-02-01,2,R Ravindra,959767,Wellington,Central Districts,0,1,0,0,0.00,1,caught,BM Tickner,854909.0,Basin Reserve,SSM,0
69884,1160525,2019-02-09,1,R Ravindra,959767,Wellington,Canterbury,6,9,0,0,66.67,1,caught,KA Jamieson,625960.0,Hagley Oval,SSM,0
80245,1197743,2019-12-15,2,R Ravindra,959767,Wellington,Canterbury,38,27,4,1,140.74,1,bowled,CE McConchie,526021.0,Hagley Oval,SSM,5
80301,1197746,2019-12-18,2,R Ravindra,959767,Wellington,Central Districts,9,7,1,0,128.57,1,caught,AY Patel,595783.0,Basin Reserve,SSM,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158190,1452615,2025-01-27,2,R Ravindra,959767,Wellington,Auckland,48,37,2,3,129.73,1,caught,AL Olliver,1419972.0,Eden Park Outer Oval,SSM,5
160548,1473440,2025-03-23,2,R Ravindra,959767,Chennai Super Kings,Mumbai Indians,65,45,2,4,144.44,0,not out,NaN,NaN,MA Chidambaram Stadium,IPL,6
160667,1473445,2025-03-28,2,R Ravindra,959767,Chennai Super Kings,Royal Challengers Bangalore,41,31,5,0,132.26,1,bowled,Yash Dayal,1159720.0,MA Chidambaram Stadium,IPL,5
160737,1473448,2025-03-30,2,R Ravindra,959767,Chennai Super Kings,Rajasthan Royals,0,4,0,0,0.00,1,caught,JC Archer,669855.0,Barsapara Cricket Stadium,IPL,0


In [33]:
updated_features_df[updated_features_df['batter_id'] == 277916].groupby('season').head(2)


,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
8679,335994,2008-04-27,1,AM Rahane,277916,Mumbai Indians,Deccan Chargers,Dr DY Patil Sports Academy,IPL,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8680,335997,2008-04-29,2,AM Rahane,277916,Mumbai Indians,Kolkata Knight Riders,Eden Gardens,IPL,0.000000,...,0.0,0.0,0.000000,0.0,7.0,0.000000,0.000000,0.000000,7.000000,0.000000
8681,392203,2009-05-01,1,AM Rahane,277916,Mumbai Indians,Kolkata Knight Riders,Buffalo Park,IPL,2.000000,...,NaN,NaN,NaN,NaN,NaN,2.000000,23.529412,2.000000,8.500000,0.500000
8682,392208,2009-05-03,1,AM Rahane,277916,Mumbai Indians,Royal Challengers Bangalore,New Wanderers Stadium,IPL,4.666667,...,10.0,10.0,125.000000,1.0,8.0,4.666667,56.000000,4.666667,8.333333,0.666667
8689,501204,2011-04-12,2,AM Rahane,277916,Rajasthan Royals,Delhi Capitals,Sawai Mansingh Stadium,IPL,18.500000,...,NaN,NaN,NaN,NaN,NaN,40.666667,119.607843,24.400000,20.400000,2.400000
8690,501214,2011-04-17,1,AM Rahane,277916,Rajasthan Royals,Kolkata Knight Riders,Eden Gardens,IPL,18.444444,...,18.0,18.0,105.882353,2.0,17.0,19.500000,111.428571,15.600000,14.000000,1.800000
8701,548309,2012-04-06,1,AM Rahane,277916,Rajasthan Royals,Punjab Kings,Sawai Mansingh Stadium,IPL,19.142857,...,NaN,NaN,NaN,NaN,NaN,21.500000,140.983607,17.200000,12.200000,2.200000
8702,548312,2012-04-08,1,AM Rahane,277916,Rajasthan Royals,Kolkata Knight Riders,Sawai Mansingh Stadium,IPL,24.400000,...,98.0,98.0,148.484848,17.0,66.0,41.750000,142.735043,33.400000,23.400000,5.200000
8722,598001,2013-04-06,1,AM Rahane,277916,Rajasthan Royals,Delhi Capitals,Feroz Shah Kotla,IPL,29.571429,...,NaN,NaN,NaN,NaN,NaN,22.600000,113.000000,22.600000,20.000000,2.600000
8723,598005,2013-04-08,1,AM Rahane,277916,Rajasthan Royals,Kolkata Knight Riders,Sawai Mansingh Stadium,IPL,29.517241,...,28.0,28.0,116.666667,3.0,24.0,24.000000,121.212121,24.000000,19.800000,3.000000


# Final Processing

In [122]:
updated_features_df['date'] = pd.to_datetime(features_df['date'])
updated_features_df.sort_values(['date', 'inns'], inplace=True)

In [123]:
min(updated_features_df.date), max(updated_features_df.date)

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-04-07 00:00:00'))

In [124]:
updated_features_df.career_avg.isnull().sum()

np.int64(4708)

In [125]:
def add_career_debut_column(df):
    """
    Adds a career_debut column to the dataframe.
    Sets value to 1 where career_avg is null (indicating league debut),
    and 0 for all other rows.

    Args:
        df: DataFrame containing batting or bowling features

    Returns:
        DataFrame with the additional career_debut column
    """
    # Create a copy to avoid modifying the original
    df = df.copy()

    # Create career_debut column based on whether career_avg is null
    df['career_debut'] = pd.isna(df['career_avg']).astype(int)

    print(f"Added career_debut column. Found {df['career_debut'].sum()} career debuts.")

    return df

In [126]:
# After creating your features dataframe
updated_features_df = add_career_debut_column(updated_features_df)

Added career_debut column. Found 4708 career debuts.


In [109]:
# stat_columns = [
#     'career_avg', 'career_rpm', 'career_sr', 'career_bpm', 'career_bfpm',
#     'inns_avg', 'inns_rpm', 'inns_sr', 'inns_bpm', 'inns_bfpm',
#     'ven_avg', 'ven_rpm', 'ven_sr', 'ven_bpm', 'ven_bfpm',
#     'ssn_avg', 'ssn_rpm', 'ssn_sr', 'ssn_bpm', 'ssn_bfpm',
#     'last5_avg', 'last5_sr', 'last5_rpm', 'last5_bfpm', 'last5_bpm'
# ]
#
# # Get indices of first appearance for each batter
# first_appearances = updated_features_df.groupby('batter_id').head(1).index
#
# # Set all stat columns to 0 for first appearances
# updated_features_df.loc[first_appearances, stat_columns] = 0

In [127]:
def convert_nan_to_zero(df):
    """
    Convert all NaN values in the dataframe to 0.

    Args:
        df: DataFrame containing features with potential NaN values

    Returns:
        DataFrame with NaN values replaced by 0
    """
    # Count NaN values before conversion
    nan_count_before = df.isna().sum().sum()

    # Make a copy to avoid modifying the original dataframe
    df_copy = df.copy()

    # Replace all NaN values with 0
    df_copy = df_copy.fillna(0)

    # Verify all NaNs have been replaced
    nan_count_after = df_copy.isna().sum().sum()

    print(f"Replaced {nan_count_before} NaN values with 0")
    print(f"Remaining NaN values: {nan_count_after}")

    return df_copy

# Usage:
updated_features_df = convert_nan_to_zero(updated_features_df)

Replaced 433725 NaN values with 0
Remaining NaN values: 0


In [128]:
updated_features_df

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm,career_debut
89289,211048,2005-02-17,1,MJ Clarke,4578,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
3184,211048,2005-02-17,1,AC Gilchrist,5390,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
68802,211048,2005-02-17,1,JR Hopes,5702,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
88201,211048,2005-02-17,1,MEK Hussey,5939,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
94361,211048,2005-02-17,1,MS Kasprowicz,6126,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52122,1473458,2025-04-07,2,HH Pandya,625371,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,28.808989,...,13.000000,118.181818,1.333333,11.000000,33.666667,146.376812,20.2,13.8,2.6,0
64033,1473458,2025-04-07,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,9.714286,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0
156655,1473458,2025-04-07,2,WG Jacks,897549,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,26.200000,...,10.666667,103.225806,1.000000,10.333333,10.000000,102.040816,10.0,9.8,1.2,0
152061,1473458,2025-04-07,2,Tilak Varma,1170265,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,40.354839,...,23.750000,113.095238,2.500000,21.000000,39.666667,120.202020,23.8,19.8,2.8,0


In [130]:
updated_features_df.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bat_features_flagged.csv", index=False)